In [19]:
import torch
import torch.nn

In [2]:
def forward(self, x: torch.Tensor) -> torch.Tensor:
    B, T, hd_k = x.size()
    h = self.num_heads
    d_k = hd_k // h # Head dimension

    # 1. Linear Projections & Reshape
    # (B, T, D) -> (B, T, h, d_k) -> (B, h, T, d_k)
    q = self.query(x).view(B, T, h, d_k).transpose(1, 2)
    k = self.key(x).view(B, T, h, d_k).transpose(1, 2)
    v = self.value(x).view(B, T, h, d_k).transpose(1, 2)

    # 2. Scaled Dot Product
    # (B, h, T, d_k) @ (B, h, d_k, T) -> (B, h, T, T)
    attention = (q @ k.transpose(-2, -1)) / (d_k**0.5)

    # 3. Masking (Casual/Triangular)
    # We create a (T, T) mask and broadcast it to (B, h, T, T)
    mask = torch.tril(torch.ones(T, T, device=x.device)) == 0
    attention = attention.masked_fill(mask, float("-inf"))

    # 4. Softmax & Weighted Values
    attention = self.softmax(attention)
    # (B, h, T, T) @ (B, h, T, d_k) -> (B, h, T, d_k)
    output = attention @ v

    # 5. Merge Heads
    # (B, h, T, d_k) -> (B, T, h, d_k) -> (B, T, D)
    return output.transpose(1, 2).contiguous().view(B, T, hd_k)

In [25]:
# B = 1
# T = 5
# hd_k = 4
# Create a tensor of shape [B, T, hd_k]
torch.manual_seed(42)
Q = torch.randint(0,16, (1,5,4))
Q

tensor([[[ 6,  3, 12, 14],
         [10,  7, 12,  4],
         [ 6,  9,  2,  6],
         [10, 10,  7,  4],
         [ 3,  7,  7,  2]]])

In [27]:
# Head 0 -> [T, d_k]
[ 6, 3 ]
[10, 7 ]
[ 6, 9 ]
[10, 10]
[ 3, 7]


[3, 7]

In [28]:
Q.view(1,5,2,2)

tensor([[[[ 6,  3],
          [12, 14]],

         [[10,  7],
          [12,  4]],

         [[ 6,  9],
          [ 2,  6]],

         [[10, 10],
          [ 7,  4]],

         [[ 3,  7],
          [ 7,  2]]]])

In [29]:
Q.view(1,5,2,2).transpose(1, 2)
# index 0 = B = 1, index 1 = T = 5, index 2 = h = 2, index 3 = d_k = 2
# Here we transpose index 1 and index 2, ie h and T
# So, [B, T, h, d_k] becomes [B, h, T, d_k]

tensor([[[[ 6,  3],
          [10,  7],
          [ 6,  9],
          [10, 10],
          [ 3,  7]],

         [[12, 14],
          [12,  4],
          [ 2,  6],
          [ 7,  4],
          [ 7,  2]]]])

In [ ]:
# In Q.view(1,5,2,2) we get outer list of size, batch size (ie 1) [, then another list of size T for the 5 words in the sentence, each element in this 5 element list is 2x2.

# The original 4 values for each word are now reshaped into a 2x2 matrix representing [num_heads, head_dim].

# So we have the 2 head values of each stacked on top of one another.